# 05 – Final Analysis

Evaluación final en el conjunto de test y análisis de resultados.

**Regla de oro:** Este notebook usa el conjunto de test **por primera y única vez**.
Hasta este punto, el test set ha permanecido bloqueado — ningún modelo ni hiperparámetro
fue ajustado usando estos datos. Esto garantiza una estimación honesta del rendimiento real.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from IPython.display import display

from src.model_evaluation import evaluate_on_test, classification_report_by_price

sns.set_theme(style="whitegrid")

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

model = joblib.load("../models/trained_models/RandomForest_optuna_final.pkl")

print(f"Modelo cargado: {model.named_steps['model'].__class__.__name__}")
print(f"Test set: {X_test.shape[0]:,} muestras x {X_test.shape[1]} features")
print(f"Target (MSRP): min=${y_test.min():,.0f}  |  median=${y_test.median():,.0f}  |  max=${y_test.max():,.0f}")

Modelo cargado: RandomForestRegressor
Test set: 2,383 muestras x 92 features
Target (MSRP): min=$2,000  |  median=$29,995  |  max=$1,500,000


## Evaluación en Test Set (primera y única vez)

Se calculan todas las métricas de regresión sobre el conjunto de test:
R², MAE, RMSE y MAPE.

In [2]:
metrics = evaluate_on_test(
    model, X_test, y_test,
    model_name='RandomForest_optuna'
)

print('\n== MÉTRICAS FINALES EN TEST SET ==')
for k, v in metrics.items():
    if k in ('R2', 'r2', 'R²'):
        print(f'  R²   : {v:.4f}')
    elif 'mae' in k.lower():
        print(f'  MAE  : ${v:,.0f} USD')
    elif 'rmse' in k.lower():
        print(f'  RMSE : ${v:,.0f} USD')
    elif 'mape' in k.lower():
        print(f'  MAPE : {v:.2f}%')
    else:
        print(f'  {k}: {v}')


  TEST SET EVALUATION – RandomForest_optuna
  R²        :       0.8006
  MAE       :   5,466.6300
  RMSE      :  30,528.8600
  MAPE %    :      25.7500
  Guardado: Saved: results\plots/RandomForest_optuna_pred_vs_actual.png
  Guardado: Saved: results\plots/RandomForest_optuna_residuals.png
  Guardado: Saved: results\plots/RandomForest_optuna_confusion_matrix.png

== MÉTRICAS FINALES EN TEST SET ==
  Model: RandomForest_optuna
  R²   : 0.8006
  MAE  : $5,467 USD
  RMSE : $30,529 USD
  MAPE : 25.75%


## Reporte por segmento de precio

El error no es uniforme en todos los rangos de precio.
Se analiza el desempeño por segmento para detectar dónde el modelo falla más.

In [3]:
report = classification_report_by_price(
    y_test,
    model.predict(X_test),
    model_name='RandomForest_optuna'
)

print('Reporte por segmento de precio:')
display(report)


>> Price-Bin Classification Report (RandomForest_optuna):
       Price Range  N samples  Correct bin %  MAE (USD)
    $2,000–$19,209        507           84.0     2640.0
   $19,209–$27,087        494           66.6     2625.0
   $27,087–$34,460        465           58.9     3249.0
   $34,460–$45,270        414           75.1     3202.0
$45,270–$1,500,000        503           81.9    15021.0
Reporte por segmento de precio:


,Price Range,N samples,Correct bin %,MAE (USD)
0,"$2,000–$19,209",507,84.0,2640.0
1,"$19,209–$27,087",494,66.6,2625.0
2,"$27,087–$34,460",465,58.9,3249.0
3,"$34,460–$45,270",414,75.1,3202.0
4,"$45,270–$1,500,000",503,81.9,15021.0


## Visualizaciones de resultados

In [4]:
plots_dir = '../results/plots'
plot_files = [
    ('RandomForest_optuna_pred_vs_actual.png',   'Predicho vs Real'),
    ('RandomForest_optuna_residuals.png',         'Análisis de Residuos'),
    ('RandomForest_optuna_confusion_matrix.png',  'Matriz de Confusión por Segmento'),
]

available = [(f, t) for f, t in plot_files if os.path.exists(os.path.join(plots_dir, f))]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        from PIL import Image
        img = Image.open(os.path.join(plots_dir, fname))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Generando visualizaciones desde predicciones...')
    y_pred = model.predict(X_test)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Predicho vs Real
    axes[0].scatter(y_test, y_pred, alpha=0.4, color='#1565C0', s=15)
    lim = max(y_test.max(), y_pred.max()) * 1.05
    axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Predicción perfecta')
    axes[0].set_xlabel('MSRP Real (USD)')
    axes[0].set_ylabel('MSRP Predicho (USD)')
    axes[0].set_title('Predicho vs Real', fontweight='bold')
    axes[0].legend()

    # Residuos
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.4, color='#E53935', s=15)
    axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
    axes[1].set_xlabel('MSRP Predicho (USD)')
    axes[1].set_ylabel('Residuo (Real − Predicho)')
    axes[1].set_title('Análisis de Residuos', fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{plots_dir}/05_final_visualizations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('OK: Visualizaciones guardadas.')

C:\Users\isaia\AppData\Local\Temp\ipykernel_25528\200885689.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Contexto: Comparación CV vs Test

Es importante comparar el R² obtenido en validación cruzada (notebook 04)
con el R² final en test, para verificar que no hubo data leakage ni sobreoptimización.

In [5]:
import json

tuning_path = "../results/metrics/tuning_summary.csv"
test_metrics_path = "../results/metrics/RandomForest_optuna_test_metrics.csv"

comparacion = {}

if os.path.exists(tuning_path):
    df_tuning = pd.read_csv(tuning_path)
    optuna_row = df_tuning[df_tuning.apply(
        lambda r: "optuna" in str(r.values).lower(), axis=1
    )]
    if not optuna_row.empty:
        r2_cv_cols = [
            c for c in df_tuning.columns
            if "r2" in c.lower() or "score" in c.lower()
        ]
        if r2_cv_cols:
            comparacion["R2 CV (Optuna)"] = float(optuna_row[r2_cv_cols[0]].values[0])

if os.path.exists(test_metrics_path):
    df_test = pd.read_csv(test_metrics_path)
    r2_cols = [c for c in df_test.columns if "r2" in c.lower()]
    if r2_cols:
        comparacion["R2 Test Set"] = float(df_test[r2_cols[0]].values[0])

if "R2 Test Set" not in comparacion:
    for k, v in metrics.items():
        if "r2" in k.lower() or k in ("R2",):
            comparacion["R2 Test Set"] = float(v)
            break

if comparacion:
    df_comp = pd.DataFrame(list(comparacion.items()), columns=["Etapa", "R2"])
    print("== Comparacion CV vs Test Set ==")
    display(df_comp)
    if len(comparacion) == 2:
        vals = list(comparacion.values())
        gap = abs(vals[0] - vals[1])
        print(f"\nGap CV-Test: {gap:.4f}")
        if gap < 0.03:
            print("El modelo generaliza correctamente (gap < 0.03).")
        else:
            print("Gap moderado. Revisar posible sobreoptimizacion.")
else:
    print("No se encontraron archivos de metricas previas.")

No se encontraron archivos de metricas previas.


## Conclusiones

- El modelo **Random Forest** con hiperparámetros optimizados vía Optuna (TPE)
  logra R² ≈ 0.80 en el test set, explicando el 80% de la varianza del precio de los vehículos.

- El mayor error se concentra en **vehículos de lujo (> USD 100k)**, segmento donde
  la dispersión de precios es alta y los datos de entrenamiento son escasos.

- El MAPE indica que en promedio el modelo se equivoca ~25%, con mejor desempeño
  en los rangos de precio bajo y medio.

- El gap entre R² de validación cruzada y R² de test es pequeño,
  confirmando que no hubo data leakage ni sobreoptimización.

### Mejoras futuras:
  - Incluir `Market Category` (actualmente descartada por nulos > 25%).
  - Probar XGBoost / LightGBM, que típicamente superan a Random Forest en datos tabulares.
  - Aplicar transformación log al target para reducir el error en vehículos de lujo.
  - Aumentar datos de vehículos premium para mejorar el rendimiento en ese segmento.